<a href="https://colab.research.google.com/github/ArturM-Science/UK-Companies-House-Daily-Tracker-AI-Analyzer/blob/main/CompaniesHouseAPIAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install required libraries
!pip install -q -U google-genai gspread oauth2client pytz

import requests
import gspread
from google.colab import auth, userdata
from google.auth import default
from google import genai
from datetime import datetime
import pytz
import random

# --- 1. CONFIGURATION & SECRETS ---
COMPANIES_HOUSE_API_KEY = userdata.get('COMPANIES_HOUSE_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
SHEET_URL = "https://docs.google.com/spreadsheets/d/1NLcKmCHJOeL10KoGipzCDbEin9JAqJBA5-ZouJCxha8/edit"

# --- 2. SET EXACT UK DATE ---
uk_timezone = pytz.timezone('Europe/London')
target_date = datetime.now(uk_timezone).strftime('%Y-%m-%d')
print(f"Fetching data for UK Date: {target_date}...\n")

# --- 3. AUTHENTICATE GOOGLE SHEETS ---
print("Authenticating with Google Sheets...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_url(SHEET_URL).sheet1

# --- 4. CALL COMPANIES HOUSE API (TWICE) ---
# Call 1: Newly Incorporated Companies
print("Fetching NEW companies...")
url_new = f"https://api.company-information.service.gov.uk/advanced-search/companies?incorporated_from={target_date}&incorporated_to={target_date}&size=5000"
response_new = requests.get(url_new, auth=(COMPANIES_HOUSE_API_KEY, ''))

new_companies = []
if response_new.status_code == 200:
    new_companies = response_new.json().get('items', [])
    for co in new_companies: co['todays_event'] = "Opened Today"
    print(f"-> Success! Found {len(new_companies)} new companies.")
else:
    print(f"-> Error fetching new companies: {response_new.status_code}")

# Call 2: Dissolved Companies
print("Fetching DISSOLVED companies...")
url_dissolved = f"https://api.company-information.service.gov.uk/advanced-search/companies?dissolved_from={target_date}&dissolved_to={target_date}&size=5000"
response_dissolved = requests.get(url_dissolved, auth=(COMPANIES_HOUSE_API_KEY, ''))

dissolved_companies = []
if response_dissolved.status_code == 200:
    dissolved_companies = response_dissolved.json().get('items', [])
    for co in dissolved_companies: co['todays_event'] = "Closed Today"
    print(f"-> Success! Found {len(dissolved_companies)} dissolved companies.\n")
else:
    print(f"-> Error fetching dissolved companies: {response_dissolved.status_code}")

all_companies = new_companies + dissolved_companies


# --- 5. PROCESS & BATCH UPLOAD TO SHEETS ---
rows_to_insert = []
all_data_for_ai = []

if all_companies:
    print("Processing data locally...")

    for co in all_companies:
        name = co.get('company_name', 'N/A')
        number = co.get('company_number', 'N/A')
        status = co.get('company_status', 'N/A')
        event = co.get('todays_event', 'N/A')

        # Calculate Age
        creation_date = co.get('date_of_creation', 'N/A')
        years_active = "N/A"
        if creation_date != 'N/A':
            try:
                c_date = datetime.strptime(creation_date, '%Y-%m-%d').date()
                t_date = datetime.now(uk_timezone).date()
                years_active = round((t_date - c_date).days / 365.25, 1)
            except ValueError:
                pass

        # Format SIC Codes
        sic_list = co.get('sic_codes', [])
        sic = ", ".join(sic_list) if sic_list else "N/A"

        # Extract Address & Postcode
        address_data = co.get('registered_office_address', {})
        postcode = address_data.get('postal_code', 'N/A')

        address_parts = [
            address_data.get('premises'), address_data.get('address_line_1'),
            address_data.get('address_line_2'), address_data.get('locality'),
            address_data.get('region'), address_data.get('country')
        ]
        full_address = ", ".join([str(part) for part in address_parts if part]) or "N/A"

        # OPTIMIZATION: Save to a Python list instead of uploading immediately
        rows_to_insert.append([target_date, name, number, status, creation_date, years_active, sic, postcode, full_address])
        all_data_for_ai.append(f"[{event}] Name: {name}, Age: {years_active} yrs, SIC: {sic}, Postcode: {postcode}")

    # OPTIMIZATION: Upload everything in one massive batch (takes 3 seconds instead of 2 hours)
    print(f"Batch uploading {len(rows_to_insert)} rows to Google Sheets...")
    sh.append_rows(rows_to_insert)
    print("Finished writing to sheets.\n")

    # --- 6. ASK GEMINI ---
    print("Sending aggregated data to Gemini for analysis...")
    client = genai.Client(api_key=GEMINI_API_KEY)

    # OPTIMIZATION: Sending 6000+ rows might trip the free-tier rate limits.
    # Here we take a random sample of 300 companies to get statistically accurate trends
    # without blowing through your quota.
    sample_size = min(300, len(all_data_for_ai))
    sampled_data = random.sample(all_data_for_ai, sample_size)
    aggregated_text = "\n".join(sampled_data)

    prompt = f"I am providing a sample of {sample_size} companies out of {len(all_companies)} total that were either incorporated or dissolved today. Summarise the key trends and give me a breakdown of the most important info. Pay attention to the geography (postcodes), industries (SIC), and how old the dissolved companies were:\n\n{aggregated_text}"

    try:
        ai_response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt,
            config={'system_instruction': "You are a helpful expert data analyst. Provide a concise, highly insightful brief."}
        )
        print("\n===============================")
        print("📊 GEMINI EXECUTIVE SUMMARY")
        print("===============================\n")
        print(ai_response.text)
    except Exception as e:
        print(f"Gemini API Error: {e}")

else:
    print("No companies were found for today's date yet.")

Fetching data for UK Date: 2026-05-12...

Authenticating with Google Sheets...
Fetching NEW companies...
-> Success! Found 1552 new companies.
Fetching DISSOLVED companies...
-> Success! Found 5000 dissolved companies.

Processing data locally...
Batch uploading 6552 rows to Google Sheets...
Finished writing to sheets.

Sending aggregated data to Gemini for analysis...
Gemini API Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: 